# Game-Start Free Building Levels

Reads `free_building_levels` weights from Google Sheets, enriches the generated static location parquet with game-start parser data, and computes where free building capacity would be available.


In [ ]:
%matplotlib inline

from pathlib import Path
import os

import matplotlib.pyplot as plt
import polars as pl
from IPython.display import Markdown, display

from eu5gameparser.load_order import LoadOrderConfig
from prosper_or_perish_population_capacity.geometry import build_location_geometry_frame
from prosper_or_perish_constructor.free_building_levels import (
    DEFAULT_DISPLAY_DECIMALS,
    SHEET_GID,
    SPREADSHEET_ID,
    compute_free_building_levels,
    contribution_category_summary,
    contribution_factor_group_summary,
    contribution_value_summary,
    explain_development_components,
    google_sheet_browser_url,
    load_game_start_development_weights,
    load_free_building_level_location_frame,
    parse_free_building_level_sheet,
    public_google_sheet_csv_url,
    read_public_google_sheet_values,
    resolve_labeling_baseline_path,
    resolve_map_data_file,
    resolve_parser_config,
    round_numeric_columns,
    summarize_free_building_levels,
    validate_sheet_values_against_game_sources,
)

pl.Config.set_float_precision(DEFAULT_DISPLAY_DECIMALS)
pl.Config.set_tbl_rows(80)
pl.Config.set_tbl_cols(30)


def find_repo(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "constructor.toml").is_file():
            return candidate
    raise FileNotFoundError("Could not find constructor.toml above current working directory")


REPO = find_repo(Path.cwd().resolve())
PROJECT = REPO / "constructor.toml"
print(f"repo={REPO}")
print(f"project={PROJECT}")


## Source Sheet

The sheet is currently readable through Google's public CSV export, so this notebook does not need OAuth for the normal read path.

- [Open the source Google Sheet](https://docs.google.com/spreadsheets/d/1d_zH-wxb9ufW6RgVZgJdGqToJ-VZP_XPS7WhhUAa18U/edit#gid=602606501)
- [Open the raw CSV export](https://docs.google.com/spreadsheets/d/1d_zH-wxb9ufW6RgVZgJdGqToJ-VZP_XPS7WhhUAa18U/export?format=csv&gid=602606501)


In [ ]:
sheet_url = google_sheet_browser_url(spreadsheet_id=SPREADSHEET_ID, gid=SHEET_GID)
csv_url = public_google_sheet_csv_url(spreadsheet_id=SPREADSHEET_ID, gid=SHEET_GID)
print(f"source_sheet={sheet_url}")
print(f"csv_export={csv_url}")

sheet_values = read_public_google_sheet_values(
    spreadsheet_id=SPREADSHEET_ID,
    gid=SHEET_GID,
)
weights = parse_free_building_level_sheet(sheet_values)

print(f"sheet_rows={len(sheet_values)} parsed_weight_rows={weights.height}")
display(round_numeric_columns(weights.sort(["factor", "value"])))


## Location Frame

This loads the existing generated location parquet, applies current location-template overlays, then enriches it with game-start ranks, market centers, capitals, roads, ports, real river levels from `rivers.png`, and computed game-start development.


In [ ]:
locations = load_free_building_level_location_frame(REPO, PROJECT)

coverage = locations.select(
    pl.len().alias("locations"),
    pl.col("has_river").sum().alias("has_river"),
    (pl.col("river_level") > 0).sum().alias("river_level_nonzero"),
    pl.col("market_center").sum().alias("market_centers"),
    pl.col("capital").sum().alias("capitals"),
    pl.col("is_port").sum().alias("ports"),
    (pl.col("road_level") > 0).sum().alias("road_locations"),
)
display(coverage)

display(
    round_numeric_columns(
        locations.select(
            "location_id",
            "location_tag",
            "province",
            "region",
            "topography",
            "vegetation",
            "river_level",
            "location_rank",
            "road_level",
            "development",
            "effective_development",
            "market_center",
            "capital",
            "province_capital",
            "is_port",
        ).head(20)
    )
)


source_validation = validate_sheet_values_against_game_sources(
    weights,
    locations,
    repo=REPO,
    project=PROJECT,
)

display(Markdown("### Sheet Value Source Validation"))
display(source_validation)


## Development Diagnostics

Game-start development is computed from `setup/start/14_development.txt`. Raw values can be negative or above 100 because the source data is additive. For free-building-level scoring, development is capped to the effective gameplay range `[0, 100]`; the raw value stays visible here so the cap is auditable.


In [ ]:
development_weights = load_game_start_development_weights(REPO, PROJECT)
development_components = explain_development_components(locations, development_weights)

development_coverage = locations.select(
    pl.len().alias("locations"),
    pl.col("development").min().alias("min_raw_development"),
    pl.col("development").median().alias("median_raw_development"),
    pl.col("development").mean().alias("mean_raw_development"),
    pl.col("development").max().alias("max_raw_development"),
    pl.col("effective_development").min().alias("min_effective_development"),
    pl.col("effective_development").mean().alias("mean_effective_development"),
    pl.col("effective_development").max().alias("max_effective_development"),
    (pl.col("development") < 0).sum().alias("raw_below_zero_locations"),
    (pl.col("development") > 100).sum().alias("raw_above_hundred_locations"),
)

negative_development = (
    development_components
    .filter(pl.col("development") < 0)
    .sort("development")
)

negative_component_columns = [
    "location_tag",
    "province",
    "region",
    "area",
    "topography",
    "vegetation",
    "climate",
    "location_rank",
    "base_development",
    "topography_development",
    "vegetation_development",
    "climate_development",
    "region_development",
    "area_development",
    "province_development",
    "location_development",
    "river_development",
    "road_development",
    "development",
    "effective_development",
]

with pl.Config(tbl_rows=80, tbl_cols=40, float_precision=DEFAULT_DISPLAY_DECIMALS):
    display(development_coverage)
    display(Markdown("### Lowest game-start development locations"))
    display(negative_development.select(negative_component_columns).head(80))


## Capacity Calculation


In [ ]:
result = compute_free_building_levels(locations, weights)
scores = result.frame

if result.diagnostics.is_empty():
    display(Markdown("No source/coverage diagnostics."))
else:
    display(result.diagnostics.sort(["severity", "factor", "value"]))

score_columns = [
    "location_id",
    "location_tag",
    "province",
    "region",
    "topography",
    "vegetation",
    "river_level",
    "location_rank",
    "road_level",
    "development",
    "effective_development",
    "free_building_levels",
]

display(Markdown("### Highest Capacity"))
display(round_numeric_columns(scores.select(score_columns).sort("free_building_levels", descending=True).head(30)))

display(Markdown("### Lowest Capacity"))
display(round_numeric_columns(scores.select(score_columns).sort("free_building_levels").head(30)))


## Contribution Shares

These tables show how much each sheet-controlled factor contributes to the global `free_building_levels` total. Development contribution uses `effective_development`, meaning raw development is capped to `[0, 100]` before multiplying by `development.per_point`. `share_of_global_total_pct` is signed and sums to 100 across factors when the global total is nonzero. `share_of_global_absolute_contribution_pct` is impact magnitude, which is easier to reason about when negative weights exist.

The value split is shown twice: first as a compact sorted table for global impact, then as full per-factor tables so Polars does not hide rows behind truncation.


In [ ]:
factor_group_contributions = contribution_factor_group_summary(scores)
category_contributions = contribution_category_summary(scores)
value_contributions = contribution_value_summary(scores)

impact_columns = [
    "factor_group",
    "factor",
    "value",
    "locations",
    "nonzero_locations",
    "total_contribution",
    "share_of_global_total_pct",
    "share_of_factor_total_pct",
    "share_of_global_absolute_contribution_pct",
    "share_of_factor_absolute_contribution_pct",
]

with pl.Config(tbl_rows=200, tbl_cols=30, float_precision=DEFAULT_DISPLAY_DECIMALS):
    display(Markdown("### Fixed vs dynamic"))
    display(factor_group_contributions)

    display(Markdown("### By sheet category / factor"))
    display(category_contributions)

    display(Markdown("### Values by global impact"))
    display(value_contributions.sort("absolute_contribution", descending=True).select(impact_columns))

    for factor in value_contributions["factor"].unique(maintain_order=True).to_list():
        display(Markdown(f"### `{factor}` value split"))
        display(
            value_contributions
            .filter(pl.col("factor") == factor)
            .sort("absolute_contribution", descending=True)
            .select(impact_columns)
        )


## Summary Tables


In [ ]:
summary_groups = [
    "super_region",
    "macro_region",
    "region",
    "province",
    "topography",
    "vegetation",
    "river_level",
    "location_rank",
    "raw_material",
]

summaries = {group: summarize_free_building_levels(scores, group) for group in summary_groups}

for group, summary in summaries.items():
    display(Markdown(f"### By `{group}`"))
    display(summary.head(25))


## Map Geometry


In [ ]:
parser_config = resolve_parser_config(REPO, PROJECT)
load_order_path = REPO / str(parser_config.get("load_order") or "constructor.load_order.toml")
profile_name = str(parser_config.get("profile") or "constructor")
profile = LoadOrderConfig.load(load_order_path).profile(profile_name)
locations_png = resolve_map_data_file(profile, "locations.png")
baseline_path = resolve_labeling_baseline_path(REPO, PROJECT)

geometry = build_location_geometry_frame(
    baseline_path=baseline_path,
    locations_png_path=locations_png,
    equator_y=3340,
)

plot_frame = scores.join(
    geometry.select("location_tag", "geometry_status", "approx_lon", "approx_lat"),
    on="location_tag",
    how="left",
).filter(pl.col("geometry_status") == "ok")

print(f"mapped_locations={plot_frame.height}")
display(round_numeric_columns(plot_frame.select("location_tag", "approx_lon", "approx_lat", "free_building_levels").head(10)))


## World Map


In [ ]:
plot_data = plot_frame.select("approx_lon", "approx_lat", "free_building_levels").to_dict(as_series=False)

fig, ax = plt.subplots(figsize=(16, 8), dpi=140)
scatter = ax.scatter(
    plot_data["approx_lon"],
    plot_data["approx_lat"],
    c=plot_data["free_building_levels"],
    s=3,
    cmap="viridis",
    linewidths=0,
    alpha=0.85,
)
ax.set_xlim(-180, 180)
ax.set_ylim(-65, 85)
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.set_title("Game-start free building levels by location")
fig.colorbar(scatter, ax=ax, label="free_building_levels")
ax.grid(alpha=0.15)
plt.show()


## Regional Aggregate Map


In [ ]:
region_plot = (
    plot_frame.group_by("region")
    .agg(
        pl.col("approx_lon").median().alias("lon"),
        pl.col("approx_lat").median().alias("lat"),
        pl.len().alias("locations"),
        pl.col("free_building_levels").mean().alias("mean_free_building_levels"),
        pl.col("free_building_levels").sum().alias("total_free_building_levels"),
    )
    .sort("total_free_building_levels", descending=True)
)
region_data = region_plot.to_dict(as_series=False)

fig, ax = plt.subplots(figsize=(16, 8), dpi=140)
sizes = [max(20, min(800, value * 1.5)) for value in region_data["locations"]]
scatter = ax.scatter(
    region_data["lon"],
    region_data["lat"],
    c=region_data["mean_free_building_levels"],
    s=sizes,
    cmap="magma",
    edgecolors="black",
    linewidths=0.25,
    alpha=0.75,
)
ax.set_xlim(-180, 180)
ax.set_ylim(-65, 85)
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.set_title("Mean free building levels by region")
fig.colorbar(scatter, ax=ax, label="mean_free_building_levels")
ax.grid(alpha=0.15)
plt.show()

display(round_numeric_columns(region_plot.head(30)))
